# Run SFT

Connect a Colab **T4** runtime and run the cells from top to bottom. The first cell clones the GitHub repo into `/content` and runs `train_sft.py` from there. Checkpoints, the data cache, and `results/` are stored in `MyDrive/rlhf/`.


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

!pip install -q datasets rouge-score matplotlib

repo = Path("/content/Summarization-from-feedback")
if not (repo / "train_sft.py").is_file():
    !git clone https://github.com/Areesha-008/Summarization-from-feedback.git /content/Summarization-from-feedback
else:
    !git -C /content/Summarization-from-feedback pull
os.chdir(repo)
root = Path.cwd()

drive_root = Path("/content/drive/MyDrive/rlhf")
for name in ("checkpoints", "data_cache", "results"):
    target = drive_root / name
    target.mkdir(parents=True, exist_ok=True)
    link = root / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        for path in link.rglob("*"):
            if path.is_file():
                dest = target / path.relative_to(link)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(path, dest)
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)

print("project:", root)
print("data cache:", (root / "data_cache" / "data.pkl").exists())
!nvidia-smi -L


In [ ]:
!python train_sft.py


In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("results/sft_samples.md", encoding="utf-8").read()))
display(Image("results/fig1_sft_loss.png"))
